In [1]:
import pandas as pd
import pm4py
import json

In [2]:
log = pm4py.read_xes("../data/BPI Challenge 2017.xes.gz")

df = pm4py.convert_to_dataframe(log)
print(f"Loaded {len(df)} events.")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pm4py/utils.py:987: UserWarning: In the current version, the import/export operation uses `rustxes` by default for importing/exporting files faster. Please uninstall `rustxes` to revert the behavior.
  warnings.warn("In the current version, the import/export operation uses `rustxes` by default for importing/exporting files faster. Please uninstall `rustxes` to revert the behavior.")


Loaded 1202267 events.


In [3]:
df.head()

,Accepted,case:LoanGoal,Selected,case:ApplicationType,CreditScore,concept:name,case:concept:name,EventOrigin,Action,OfferID,time:timestamp,org:resource,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,MonthlyCost,lifecycle:transition,OfferedAmount,EventID
0,None,Existing loan takeover,None,New credit,NaN,A_Create Application,Application_652823628,Application,Created,None,2016-01-01 09:51:15.304000+00:00,User_1,20000.0,NaN,NaN,NaN,complete,NaN,Application_652823628
1,None,Existing loan takeover,None,New credit,NaN,A_Submitted,Application_652823628,Application,statechange,None,2016-01-01 09:51:15.352000+00:00,User_1,20000.0,NaN,NaN,NaN,complete,NaN,ApplState_1582051990
2,None,Existing loan takeover,None,New credit,NaN,W_Handle leads,Application_652823628,Workflow,Created,None,2016-01-01 09:51:15.774000+00:00,User_1,20000.0,NaN,NaN,NaN,schedule,NaN,Workitem_1298499574
3,None,Existing loan takeover,None,New credit,NaN,W_Handle leads,Application_652823628,Workflow,Deleted,None,2016-01-01 09:52:36.392000+00:00,User_1,20000.0,NaN,NaN,NaN,withdraw,NaN,Workitem_1673366067
4,None,Existing loan takeover,None,New credit,NaN,W_Complete application,Application_652823628,Workflow,Created,None,2016-01-01 09:52:36.403000+00:00,User_1,20000.0,NaN,NaN,NaN,schedule,NaN,Workitem_1493664571


In [4]:
#Filter only for completed events so that we can only consider A_Created (Complete) not A_created (Start) and (Complete)
if 'lifecycle:transition' in df.columns:
    df = df[df['lifecycle:transition'].str.lower() == 'complete']
df.head()

,Accepted,case:LoanGoal,Selected,case:ApplicationType,CreditScore,concept:name,case:concept:name,EventOrigin,Action,OfferID,time:timestamp,org:resource,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,MonthlyCost,lifecycle:transition,OfferedAmount,EventID
0,None,Existing loan takeover,None,New credit,NaN,A_Create Application,Application_652823628,Application,Created,None,2016-01-01 09:51:15.304000+00:00,User_1,20000.0,NaN,NaN,NaN,complete,NaN,Application_652823628
1,None,Existing loan takeover,None,New credit,NaN,A_Submitted,Application_652823628,Application,statechange,None,2016-01-01 09:51:15.352000+00:00,User_1,20000.0,NaN,NaN,NaN,complete,NaN,ApplState_1582051990
5,None,Existing loan takeover,None,New credit,NaN,A_Concept,Application_652823628,Application,statechange,None,2016-01-01 09:52:36.413000+00:00,User_1,20000.0,NaN,NaN,NaN,complete,NaN,ApplState_642383566
8,None,Existing loan takeover,None,New credit,NaN,A_Accepted,Application_652823628,Application,statechange,None,2016-01-02 11:23:04.299000+00:00,User_52,20000.0,NaN,NaN,NaN,complete,NaN,ApplState_99568828
9,True,Existing loan takeover,True,New credit,979.0,O_Create Offer,Application_652823628,Offer,Created,None,2016-01-02 11:29:03.994000+00:00,User_52,20000.0,20000.0,44.0,498.29,complete,20000.0,Offer_148581083


In [5]:
#Now df is sorted by time and case
df = df.sort_values(by=['case:concept:name', 'time:timestamp'])
df.head(20)

,Accepted,case:LoanGoal,Selected,case:ApplicationType,CreditScore,concept:name,case:concept:name,EventOrigin,Action,OfferID,time:timestamp,org:resource,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,MonthlyCost,lifecycle:transition,OfferedAmount,EventID
686058,None,"Other, see explanation",None,New credit,NaN,A_Create Application,Application_1000086665,Application,Created,None,2016-08-03 15:57:21.673000+00:00,User_1,5000.0,NaN,NaN,NaN,complete,NaN,Application_1000086665
686059,None,"Other, see explanation",None,New credit,NaN,A_Submitted,Application_1000086665,Application,statechange,None,2016-08-03 15:57:21.734000+00:00,User_1,5000.0,NaN,NaN,NaN,complete,NaN,ApplState_161925113
686063,None,"Other, see explanation",None,New credit,NaN,A_Concept,Application_1000086665,Application,statechange,None,2016-08-03 15:58:28.299000+00:00,User_1,5000.0,NaN,NaN,NaN,complete,NaN,ApplState_385184570
686066,None,"Other, see explanation",None,New credit,NaN,A_Accepted,Application_1000086665,Application,statechange,None,2016-08-05 13:57:07.419000+00:00,User_5,5000.0,NaN,NaN,NaN,complete,NaN,ApplState_856156982
686067,True,"Other, see explanation",False,New credit,0.0,O_Create Offer,Application_1000086665,Offer,Created,None,2016-08-05 13:59:57.320000+00:00,User_5,5000.0,5000.0,22.0,241.28,complete,5000.0,Offer_410892064
686068,None,"Other, see explanation",None,New credit,NaN,O_Created,Application_1000086665,Offer,statechange,Offer_410892064,2016-08-05 13:59:58.162000+00:00,User_5,5000.0,NaN,NaN,NaN,complete,NaN,OfferState_2071773136
686069,None,"Other, see explanation",None,New credit,NaN,O_Sent (mail and online),Application_1000086665,Offer,statechange,Offer_410892064,2016-08-05 14:01:23.264000+00:00,User_5,5000.0,NaN,NaN,NaN,complete,NaN,OfferState_388269514
686073,None,"Other, see explanation",None,New credit,NaN,A_Complete,Application_1000086665,Application,statechange,None,2016-08-05 14:01:23.288000+00:00,User_5,5000.0,NaN,NaN,NaN,complete,NaN,ApplState_1103800866
686077,None,"Other, see explanation",None,New credit,NaN,A_Cancelled,Application_1000086665,Application,statechange,None,2016-09-05 06:00:36.710000+00:00,User_1,5000.0,NaN,NaN,NaN,complete,NaN,ApplState_544389273
686078,None,"Other, see explanation",None,New credit,NaN,O_Cancelled,Application_1000086665,Offer,statechange,Offer_410892064,2016-09-05 06:00:36.829000+00:00,User_1,5000.0,NaN,NaN,NaN,complete,NaN,OfferState_1258287612


Finding which activities are end activities?

In [6]:
#New column for the next activity
df['next_activity'] = df.groupby('case:concept:name')['concept:name'].shift(-1)
next_activity_none = df[df['next_activity'].isna()]
end_events = next_activity_none
print(next_activity_none['concept:name'].value_counts().head(10))

concept:name
O_Cancelled                 14051
A_Pending                   10288
W_Validate application       4036
O_Refused                    1977
W_Call incomplete files       631
W_Call after offers           135
A_Cancelled                   119
W_Assess potential fraud       93
W_Complete application         63
A_Complete                     30
Name: count, dtype: int64


Filter the rows where there isn't a next_activity available (end activity) NaN 

In [7]:
transitions = df.dropna(subset=['next_activity'])
transitions['next_activity'].isna().sum()

np.int64(0)

What are the lines where the activity (concept:name) == next_activity --> loops

In [8]:
self_loops = transitions[transitions['concept:name'] == transitions['next_activity']]
num_of_self_loops = len(self_loops)
num_of_self_loops

8676

In [9]:
cols_to_show = ['case:concept:name', 'concept:name', 'lifecycle:transition', 'next_activity']
check_df = self_loops.copy()
check_df['next_lifecycle'] = df.groupby('case:concept:name')['lifecycle:transition'].shift(-1)
check_df[cols_to_show + ['next_lifecycle']]

,case:concept:name,concept:name,lifecycle:transition,next_activity,next_lifecycle
45028,Application_1000691650,O_Sent (mail and online),complete,O_Sent (mail and online),complete
45043,Application_1000691650,O_Sent (mail and online),complete,O_Sent (mail and online),complete
45046,Application_1000691650,O_Cancelled,complete,O_Cancelled,complete
45047,Application_1000691650,O_Cancelled,complete,O_Cancelled,complete
45048,Application_1000691650,O_Cancelled,complete,O_Cancelled,complete
...,...,...,...,...,...
1084755,Application_999464866,O_Cancelled,complete,O_Cancelled,complete
1084756,Application_999464866,O_Cancelled,complete,O_Cancelled,complete
373773,Application_999544538,O_Cancelled,complete,O_Cancelled,complete
373774,Application_999544538,O_Cancelled,complete,O_Cancelled,complete


In [10]:
print(self_loops['concept:name'].value_counts())

concept:name
O_Cancelled                 4429
O_Sent (mail and online)    3111
O_Refused                    975
O_Sent (online only)          85
O_Returned                    65
W_Assess potential fraud       5
W_Validate application         3
A_Incomplete                   3
Name: count, dtype: int64


## Self-loop handling

Raw self-loops (same activity consecutive in a case) fall into two categories:

1. **True modelling artefacts** (e.g. W_Validate → W_Validate): drop them.  
2. **Concurrent offer events** (O_Cancelled → O_Cancelled, O_Sent → O_Sent, O_Refused → O_Refused): these represent multiple simultaneous offers that are processed sequentially in the log. In our sequential Petri-net model there is no notion of parallel offers, so the closest equivalent is looping back through **O_Create Offer**. Instead of discarding these rows we **redirect** them to their sequential equivalent.

| Self-loop | Redirect to | Reasoning |
|-----------|-------------|-----------|
| O_Cancelled → O_Cancelled | O_Create Offer | Another (queued) offer is opened after cancellation |
| O_Sent → O_Sent | O_Create Offer | Another offer notification is triggered |
| O_Refused → O_Refused | W_Validate application | Most common real exit after refusal (84 % of branching transitions) |

In [11]:
# Offer-level self-loops: redirect to sequential equivalent instead of dropping
self_loop_redirect = {
    'O_Cancelled':            'O_Create Offer',
    'O_Sent (mail and online)': 'O_Create Offer',
    'O_Refused':              'W_Validate application',
}

redirected_loops = self_loops[self_loops['concept:name'].isin(self_loop_redirect)].copy()
redirected_loops = redirected_loops.copy()
redirected_loops['next_activity'] = redirected_loops['concept:name'].map(self_loop_redirect)

# True self-loops (not redirected) are dropped
truly_removed = self_loops[~self_loops['concept:name'].isin(self_loop_redirect)]

branching_transitions = pd.concat([
    transitions[transitions['concept:name'] != transitions['next_activity']],
    redirected_loops
], ignore_index=True)

print(f"Redirected self-loops : {len(redirected_loops):>6,}  {redirected_loops['concept:name'].value_counts().to_dict()}")
print(f"Truly removed loops   : {len(truly_removed):>6,}  {truly_removed['concept:name'].value_counts().to_dict()}")
print(f"Total branching trans : {len(branching_transitions):>6,}")

Redirected self-loops :  8,515  {'O_Cancelled': 4429, 'O_Sent (mail and online)': 3111, 'O_Refused': 975}
Truly removed loops   :    161  {'O_Sent (online only)': 85, 'O_Returned': 65, 'W_Assess potential fraud': 5, 'W_Validate application': 3, 'A_Incomplete': 3}
Total branching trans : 443,636


In [12]:
containing_any_self_loop = branching_transitions[branching_transitions['concept:name'] == branching_transitions['next_activity']]
len(containing_any_self_loop)

0

In [13]:
all_branching = branching_transitions.groupby(['concept:name', 'next_activity']).size().sort_values(ascending=False)
branching_df = all_branching.reset_index(name='count')
sorted_branching_df = branching_df.sort_values(by=['concept:name', 'count'], ascending=[True, False])
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(sorted_branching_df)

,concept:name,next_activity,count
2,A_Accepted,O_Create Offer,31504
125,A_Accepted,W_Complete application,3
129,A_Accepted,W_Assess potential fraud,2
15,A_Cancelled,O_Cancelled,10270
89,A_Cancelled,W_Call incomplete files,24
104,A_Cancelled,W_Call after offers,11
109,A_Cancelled,W_Complete application,7
8,A_Complete,A_Validating,18339
16,A_Complete,A_Cancelled,8034
25,A_Complete,O_Create Offer,4135


Probabilities

In [14]:
counts = branching_transitions.groupby(['concept:name', 'next_activity']).size().unstack(fill_value=0)
probabilities = counts.div(counts.sum(axis=1), axis=0)
decision_prob_rules = probabilities.to_dict(orient='index')
decision_prob_rules

{'A_Accepted': {'A_Accepted': 0.0,
  'A_Cancelled': 0.0,
  'A_Complete': 0.0,
  'A_Concept': 0.0,
  'A_Denied': 0.0,
  'A_Incomplete': 0.0,
  'A_Pending': 0.0,
  'A_Submitted': 0.0,
  'A_Validating': 0.0,
  'O_Accepted': 0.0,
  'O_Cancelled': 0.0,
  'O_Create Offer': 0.9998413151797899,
  'O_Created': 0.0,
  'O_Refused': 0.0,
  'O_Returned': 0.0,
  'O_Sent (mail and online)': 0.0,
  'O_Sent (online only)': 0.0,
  'W_Assess potential fraud': 6.347392808403948e-05,
  'W_Call after offers': 0.0,
  'W_Call incomplete files': 0.0,
  'W_Complete application': 9.521089212605922e-05,
  'W_Handle leads': 0.0,
  'W_Validate application': 0.0},
 'A_Cancelled': {'A_Accepted': 0.0,
  'A_Cancelled': 0.0,
  'A_Complete': 0.0,
  'A_Concept': 0.0,
  'A_Denied': 0.0,
  'A_Incomplete': 0.0,
  'A_Pending': 0.0,
  'A_Submitted': 0.0,
  'A_Validating': 0.0,
  'O_Accepted': 0.0,
  'O_Cancelled': 0.9959270752521334,
  'O_Create Offer': 0.0,
  'O_Created': 0.0,
  'O_Refused': 0.0,
  'O_Returned': 0.0,
  'O_Sen

Save the probabilities as a csv file

In [15]:
output_file = "decision_prob_rules.json"
with open(output_file, "w") as f:
    json.dump(decision_prob_rules, f, indent=4)

Debug the test_output.csv

In [16]:
import pandas as pd

df = pd.read_csv('/Users/zeynepcetin/bppso-groupwork-1/test_output.csv')
decision_step = 'A_Cancelled'

df = df.sort_values(by=['case:concept:name', 'time:timestamp'])
df['next'] = df.groupby('case:concept:name')['concept:name'].shift(-1)
print(df[df['concept:name'] == decision_step]['next'].value_counts(normalize=True))

FileNotFoundError: [Errno 2] No such file or directory: '/Users/zeynepcetin/bppso-groupwork-1/test_output.csv'

In [ ]:
import pandas as pd

df = pd.read_csv('/Users/zeynepcetin/bppso-groupwork-1/test_output.csv')

if 'lifecycle:transition' in df.columns:
    df = df[df['lifecycle:transition'].str.lower() == 'complete']

df = df.sort_values(by=['case:concept:name', 'time:timestamp'])
df['next_activity'] = df.groupby('case:concept:name')['concept:name'].shift(-1)

unique_activities = df['concept:name'].unique()

for activity in unique_activities:
    transitions = df[df['concept:name'] == activity]['next_activity']
    
    if transitions.dropna().empty:
        continue

    probs = transitions.value_counts(normalize=True)
    
    print(f"'{activity}':")
    for target, prob in probs.items():
        print(f"    {target}: {prob:.5f}")
    print("")

'A_Create Application':
    A_Submitted: 1.00000

'A_Submitted':
    W_Handle leads: 1.00000

'W_Handle leads':
    A_Concept: 1.00000

'A_Concept':
    W_Complete application: 1.00000

'W_Complete application':
    A_Accepted: 1.00000

'A_Accepted':
    O_Create Offer: 1.00000

'O_Create Offer':
    O_Created: 1.00000

'O_Created':
    O_Sent (mail and online): 0.51823
    O_Cancelled: 0.48177

'O_Sent (mail and online)':
    A_Complete: 1.00000

'A_Complete':
    W_Call after offers: 1.00000

'W_Call after offers':
    O_Create Offer: 0.51070
    A_Validating: 0.26911
    A_Cancelled: 0.11621
    O_Cancelled: 0.10398

'O_Cancelled':
    O_Cancelled: 0.49728
    O_Create Offer: 0.44429
    A_Cancelled: 0.05842

'A_Validating':
    W_Validate application: 1.00000

'W_Validate application':
    O_Returned: 1.00000

'O_Returned':
    O_Accepted: 0.50526
    A_Incomplete: 0.41053
    A_Denied: 0.08421

'A_Incomplete':
    W_Call incomplete files: 1.00000

'W_Call incomplete files':
    O_

+ Where the Decision Manager works: 
A_Accepted, O_Create Offer, O_Accepted, A_Denied, A_Cancelled
- middle
A_Pending, W_Call after offers, A_Complete, O_Sent (mail and online), A_Validating
- problematic
A_Create Application, A_Submitted, A_Concept, W_C a, O_Created, W_Validate application
A_Pending

In [ ]:
df_out = pd.read_csv('/Users/zeynepcetin/bppso-groupwork-1/test_output.csv')

if 'lifecycle:transition' in df_out.columns:
    df_out = df_out[df_out['lifecycle:transition'].str.lower() == 'complete']

distinct_activities = sorted(df_out['concept:name'].dropna().unique())

print(f"Number of distinct activities: {len(distinct_activities)}\n")
for a in distinct_activities:
    print(a)

Number of distinct activities: 22

A_Accepted
A_Cancelled
A_Complete
A_Concept
A_Create Application
A_Denied
A_Incomplete
A_Pending
A_Submitted
A_Validating
O_Accepted
O_Cancelled
O_Create Offer
O_Created
O_Refused
O_Returned
O_Sent (mail and online)
W_Call after offers
W_Call incomplete files
W_Complete application
W_Handle leads
W_Validate application


In [ ]:
activity_counts = (
    df_out['concept:name']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'activity', 'concept:name': 'count'})
)
display(activity_counts)


,count,count
0,O_Cancelled,784
1,O_Create Offer,631
2,O_Created,631
3,O_Sent (mail and online),327
4,A_Complete,327
5,W_Call after offers,327
6,A_Create Application,100
7,W_Handle leads,100
8,A_Concept,100
9,W_Complete application,100
